# Classification

## Configuration

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.ConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

In [ ]:
from utils.csv import csv_col_dict
ACTION_UNITS = csv_col_dict(
    conf['Features']['action-units'].strip().split('\n')
)['name']

## Data import

In [ ]:
SOURCE = '../.data/processed.pkl'
import pandas as pd
data = pd.read_pickle(SOURCE)
data

## Generate Datasets

Windowed

In [ ]:
CLASSES = ['state_understanding', 'state_confusion']
import numpy as np
def to_windows(df:pd.DataFrame, window_size=50, stride=25):
    # parameter validation
    window_size=max(window_size, 1) # ensure window size of at least 1
    stride=max(stride, 1) # ensure stride of at least 1

    # decompose to recording-level
    for p, participant in df.groupby('token'):
        for c, recording in participant.groupby('condition'):
            # ensure ascending order
            recording = recording.sort_values('frame').reset_index(drop=True)

            # identify coninous segments
            segment_idcs = (recording['frame'].diff().fillna(1) != 1).cumsum()
            _, segments = zip(*recording.groupby(segment_idcs))
            for segment in segments:
                n_frames = len(segment)
                # skip short segments
                if n_frames < window_size:
                    continue
                
                # generate windows
                for start in range(0, n_frames, stride):
                    end = start+window_size
                    # skip if not enough frames remain
                    if end > n_frames:
                        continue
                    
                    # extract window
                    yield segment.iloc[start:end]

            # DEBUG Process single recording only
            #return

def to_set(windows, features_x, features_y):
    data_x = list()
    data_y = list()
    for window in windows:
        data_x.append(window[features_x].astype(float))
        data_y.append(window[features_y])
    
    return np.array(data_x), np.array(data_y)


def to_training_data(df:pd.DataFrame, conditions:list=['stress', 'neutral'], 
                     window_duration:float=2, fps:float=50):
    
    window_size = window_duration*fps
    windows = to_windows(
        df[df['condition'].isin(list(conditions))],
        window_size=int(window_size),
        stride=int(window_size/2)
    )

    x = list()
    y = list()
    for window in windows:
        x.append(np.array(window[ACTION_UNITS].astype(float)))
        y.append(np.array(window[CLASSES].astype(float)))

    return np.array(x), np.array(y)
        

In [ ]:
DURATIONS = (1,2,4,6,8)
DIR_OUT = '../.data/'

for d in DURATIONS:
    for c in ('neutral', 'stress'):
        x, y, = to_training_data(
            data, conditions=[c],
            window_duration=d
        )
        np.savez(
            file=os.path.join(DIR_OUT, f"train_{c}_{d}s.npz"),
            x=x, y=y
        )

Single Frame

In [ ]:
for c in ('neutral', 'stress'):
    df = data[data['condition'] == c]
    df = df[df['success']]
    x = np.expand_dims(np.array(df[ACTION_UNITS].astype(float)), axis=1)
    y = np.expand_dims(np.array(df[CLASSES].astype(float)), axis=1)
    np.savez(
        file=os.path.join(DIR_OUT, f"train_{c}_0s.npz"),
        x=x, y=y
    )